# Cyclone Prediction Model
## File: models/train_cyclone.ipynb

**Run karo:** Kernel → Restart & Run All
**Output:** `cyclone_model.pkl` → Copy karo `backend/` folder mein

**Cyclone season India:**
- Bay of Bengal: October–December
- Arabian Sea: May–June

## Step 1 — Libraries

In [1]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, f1_score,
    roc_auc_score)
print("Ready!")

Ready!


## Step 2 — Dataset + Train + Save

In [2]:
def generate_cyclone_data(n=5000, seed=43):
    
    #Cyclone dataset — Bay of Bengal + Arabian Sea focus.
    #Key factors: hot sea, low pressure, high winds, low wind shear.
    
    #Real sources:
    #- IMD Cyclone Warning: https://mausam.imd.gov.in
    #- INCOIS Ocean Data: https://incois.gov.in

    np.random.seed(seed)
    
    sst        = np.random.uniform(24, 32, n)     # sea surface temp
    wind_speed = np.random.uniform(20, 220, n)
    pressure   = np.random.uniform(900, 1013, n)  # low = cyclone
    humidity   = np.random.uniform(50, 100, n)
    dist_coast = np.random.exponential(150, n).clip(5, 1000)
    latitude   = np.random.uniform(5, 25, n)
    month      = np.random.randint(1, 13, n)
    wind_shear = np.random.uniform(0, 40, n)       # low = favorable
    ohc        = np.random.uniform(20, 120, n)     # ocean heat content
    past_wind  = wind_speed * np.random.uniform(0.6, 1.4, n)
    
    p  = 0.01
    p += 0.30 * ((sst - 24) / 8).clip(0, 1)
    p += 0.25 * (wind_speed / 220)
    p += 0.20 * ((1013 - pressure) / 113)
    p += 0.10 * (ohc / 120)
    p += 0.08 * (1 - wind_shear / 40)
    p += 0.07 * (humidity / 100)
    p += np.where(np.isin(month, [10,11,12,5,6]), 0.08, 0)
    p -= 0.05 * (dist_coast / 1000)
    p += np.random.normal(0, 0.04, n)
    p  = p.clip(0, 1)
    cyclone = (p > np.percentile(p, 83)).astype(int)
    
    df = pd.DataFrame({
        'sea_surface_temp':sst.round(2),'wind_speed_kmh':wind_speed.round(1),
        'pressure_hpa':pressure.round(1),'humidity':humidity.round(1),
        'distance_coast_km':dist_coast.round(1),'latitude':latitude.round(3),
        'month':month,'wind_shear':wind_shear.round(2),
        'ocean_heat_content':ohc.round(2),'past_wind_24h':past_wind.round(1),
        'cyclone':cyclone
    })
    return df

FEATURES = [
    'sea_surface_temp','wind_speed_kmh','pressure_hpa','humidity',
    'distance_coast_km','latitude','month','wind_shear',
    'ocean_heat_content','past_wind_24h',
    'pressure_drop','wind_intensification','thermal_energy'
]

def engineer(df):
    df = df.copy()
    df['pressure_drop']         = 1013 - df['pressure_hpa']
    df['wind_intensification']  = df['wind_speed_kmh'] / (df['past_wind_24h'] + 1)
    df['thermal_energy']        = df['sea_surface_temp'] * df['ocean_heat_content'] / 100
    return df

df     = generate_cyclone_data()
df_eng = engineer(df)
X      = df_eng[FEATURES]; y = df_eng['cyclone']
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

model = RandomForestClassifier(n_estimators=200,max_depth=15,
    class_weight='balanced',random_state=42,n_jobs=-1)
model.fit(X_tr, y_tr)
pred  = model.predict(X_te)
proba = model.predict_proba(X_te)[:,1]

print("CYCLONE MODEL RESULTS")
print(f"F1-Score : {f1_score(y_te, pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_te, proba):.4f}")
print()
print(classification_report(y_te, pred, target_names=['No Cyclone','Cyclone']))

with open('cyclone_model.pkl','wb') as f:
    pickle.dump({'model':model,'features':FEATURES,'version':'1.0','disaster':'cyclone'},f)
print("cyclone_model.pkl saved! Copy to backend/ folder.")

CYCLONE MODEL RESULTS
F1-Score : 0.6806
ROC-AUC  : 0.9550

              precision    recall  f1-score   support

  No Cyclone       0.92      0.98      0.95       830
     Cyclone       0.83      0.58      0.68       170

    accuracy                           0.91      1000
   macro avg       0.87      0.78      0.81      1000
weighted avg       0.90      0.91      0.90      1000

cyclone_model.pkl saved! Copy to backend/ folder.
